# 第14章 Actor-Critic方法

## 目录
1. 环境搭建
2. Actor-Critic框架
3. A2C/A3C算法
4. 编程题

---
## 1. 环境搭建

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
print("环境搭建完成!")

---
## 2. Actor-Critic框架

In [ ]:
class ActorCritic:
    """Actor-Critic算法"""
    def __init__(self, n_states, n_actions, alpha_a=0.01, alpha_c=0.1, gamma=0.99):
        self.actor = np.zeros((n_states, n_actions))
        self.critic = np.zeros(n_states)
        self.alpha_a, self.alpha_c, self.gamma = alpha_a, alpha_c, gamma
    
    def get_policy(self, s):
        exp_a = np.exp(self.actor[s] - np.max(self.actor[s]))
        return exp_a / np.sum(exp_a)
    
    def get_value(self, s): return self.critic[s]
    
    def select_action(self, s):
        return np.random.choice(len(self.actor[s]), p=self.get_policy(s))
    
    def update(self, s, a, r, ns, done):
        target = r + self.gamma * (0 if done else self.critic[ns])
        td_err = target - self.critic[s]
        self.critic[s] += self.alpha_c * td_err
        pi = self.get_policy(s)
        self.actor[s, a] += self.alpha_a * td_err * (1 - pi[a])

print("Actor-Critic框架: 策略(actor)+价值(critic)")

---
## 3. A2C/A3C算法

In [ ]:
class A2C:
    """Advantage Actor-Critic"""
    def __init__(self, n_states, n_actions, alpha=0.001, gamma=0.99):
        self.actor = np.random.randn(n_states, n_actions) * 0.01
        self.critic = np.zeros(n_states)
        self.alpha, self.gamma = alpha, gamma
    
    def get_advantage(self, s, r, ns, done):
        target = r + self.gamma * (0 if done else self.critic[ns])
        return target - self.critic[s]
    
    def update(self, s, a, r, ns, done):
        advantage = self.get_advantage(s, r, ns, done)
        self.critic[s] += self.alpha * 10 * advantage
        exp_a = np.exp(self.actor[s] - np.max(self.actor[s]))
        pi = exp_a / np.sum(exp_a)
        self.actor[s, a] += self.alpha * advantage * (1 - pi[a])

class A3C:
    """异步Actor-Critic (简化版)"""
    def __init__(self, n_workers=4, **kwargs):
        self.workers = [A2C(**kwargs) for _ in range(n_workers)]
        self.global_ac = A2C(**kwargs)
    
    def async_update(self, worker_id, grads):
        pass

print("A2C: 使用优势函数, A3C: 异步并行")

---
## 4. 编程题

### 编程题1：实现A2C算法在CartPole环境中的训练

In [ ]:
class SimpleCartPole:
    """简化CartPole"""
    def __init__(self):
        self.state = np.zeros(4)
    
    def reset(self):
        self.state = np.random.randn(4) * 0.1
        return self.state.copy()
    
    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = 10.0 if action == 1 else -10.0
        costheta, sintheta = np.cos(theta), np.sin(theta)
        temp = force + 0.1 * 0.5 * theta_dot ** 2 * sintheta
        theta_acc = (9.8 * sintheta - costheta * temp) / (0.5 * (4.0/3.0 - 0.1 * costheta ** 2))
        x_acc = temp - 0.1 * theta_acc * costheta
        x += 0.02 * x_dot + 0.02 * x_acc
        x_dot += 0.02 * x_acc
        theta += 0.02 * theta_dot + 0.02 * theta_acc
        theta_dot += 0.02 * theta_acc
        self.state = np.array([x, x_dot, theta, theta_dot])
        done = x < -2.4 or x > 2.4 or theta < -0.2095 or theta > 0.2095
        return self.state.copy(), (0 if done else 1.0), done

class A2CAgent:
    """A2C智能体"""
    def __init__(self, n_states, n_actions, alpha=0.001, gamma=0.99):
        self.actor = np.random.randn(n_states, n_actions) * 0.01
        self.critic = np.zeros(n_states)
        self.alpha, self.gamma = alpha, gamma
        self.n_states = n_states
    
    def get_state_idx(self, state):
        idx = int(np.clip(state[0] * 2 + 4, 0, self.n_states - 1))
        return idx
    
    def select_action(self, state):
        s = self.get_state_idx(state)
        exp_a = np.exp(self.actor[s] - np.max(self.actor[s]))
        return np.random.choice(len(exp_a), p=exp_a / np.sum(exp_a))
    
    def update(self, s, a, r, ns, done):
        s_idx = self.get_state_idx(s)
        ns_idx = self.get_state_idx(ns)
        target = r + self.gamma * (0 if done else self.critic[ns_idx])
        advantage = target - self.critic[s_idx]
        self.critic[s_idx] += self.alpha * advantage
        exp_a = np.exp(self.actor[s_idx])
        pi = exp_a / np.sum(exp_a)
        self.actor[s_idx, a] += self.alpha * advantage * (1 - pi[a])

def train_a2c():
    """训练A2C"""
    agent = A2CAgent(n_states=8, n_actions=2)
    env = SimpleCartPole()
    rewards_history = []
    
    for ep in range(200):
        s = env.reset()
        total_reward = 0
        for _ in range(500):
            a = agent.select_action(s)
            ns, r, done = env.step(a)
            agent.update(s, a, r, ns, done)
            s = ns
            total_reward += r
            if done: break
        rewards_history.append(total_reward)
    
    print("A2C训练结果:")
    print(f"  最终回报: {rewards_history[-1]}")
    print(f"  平均回报(最后10局): {np.mean(rewards_history[-10:]):.1f}")

train_a2c()

---

### 编程题2：实现A3C异步训练框架

In [ ]:
import threading
import queue

class GlobalA3C:
    """全局A3C共享器"""
    def __init__(self, n_states, n_actions):
        self.actor = np.random.randn(n_states, n_actions) * 0.01
        self.critic = np.zeros(n_states)
        self.lock = threading.Lock()
    
    def pull(self):
        with self.lock:
            return self.actor.copy(), self.critic.copy()
    
    def push(self, actor_grad, critic_grad):
        with self.lock:
            self.actor += actor_grad
            self.critic += critic_grad

class A3CWorker:
    """A3C工作线程"""
    def __init__(self, worker_id, global_ac, env_fn):
        self.worker_id = worker_id
        self.global_ac = global_ac
        self.env_fn = env_fn
        self.gamma = 0.99
    
    def run(self, n_steps=50):
        actor, critic = self.global_ac.pull()
        n_states, n_actions = actor.shape
        
        env = self.env_fn()
        s = env.reset()
        
        for _ in range(n_steps):
            exp_a = np.exp(actor[int(s[0]*2+4)%8] - np.max(actor[int(s[0]*2+4)%8]))
            pi = exp_a / np.sum(exp_a)
            a = np.random.choice(n_actions, p=pi)
            ns, r, done = env.step(a)
            
            target = r + self.gamma * (0 if done else critic[int(ns[0]*2+4)%8])
            advantage = target - critic[int(s[0]*2+4)%8]
            
            actor_grad = np.zeros_like(actor)
            critic_grad = np.zeros_like(critic)
            s_idx = int(s[0]*2+4)%8
            actor_grad[s_idx, a] = 0.01 * advantage * (1 - pi[a])
            critic_grad[s_idx] = 0.01 * advantage
            
            self.global_ac.push(actor_grad, critic_grad)
            s = ns
            if done: s = env.reset()

def test_a3c():
    """测试A3C"""
    global_ac = GlobalA3C(n_states=8, n_actions=2)
    
    def make_env():
        return SimpleCartPole()
    
    workers = [A3CWorker(i, global_ac, make_env) for i in range(4)]
    
    print("A3C初始化完成:")
    print(f"  工作线程数: {len(workers)}")
    print(f"  全局参数形状: actor={global_ac.actor.shape}, critic={global_ac.critic.shape}")

test_a3c()

---

### 编程题3：实现带熵正则化的Actor-Critic

In [ ]:
class EntropyAC:
    """带熵正则化的Actor-Critic"""
    def __init__(self, n_states, n_actions, alpha=0.001, beta=0.01, gamma=0.99):
        self.actor = np.zeros((n_states, n_actions))
        self.critic = np.zeros(n_states)
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
    
    def entropy(self, state):
        exp_a = np.exp(self.actor[state] - np.max(self.actor[state]))
        pi = exp_a / np.sum(exp_a)
        return -np.sum(pi * np.log(pi + 1e-8))
    
    def update(self, s, a, r, ns, done):
        target = r + self.gamma * (0 if done else self.critic[ns])
        advantage = target - self.critic[s]
        self.critic[s] += self.alpha * advantage
        exp_a = np.exp(self.actor[s] - np.max(self.actor[s]))
        pi = exp_a / np.sum(exp_a)
        ent = self.entropy(s)
        self.actor[s, a] += self.alpha * (advantage + self.beta * ent) * (1 - pi[a])

class GridEnvAC:
    """网格世界环境"""
    def __init__(self, size=4):
        self.size = size
        self.state = 0
    def reset(self): self.state = 0; return self.state
    def step(self, a):
        if a == 0: self.state = max(0, self.state - 1)
        elif a == 1: self.state = min(self.size * self.size - 1, self.state + 1)
        r = 1.0 if self.state == self.size * self.size - 1 else -0.01
        done = self.state == self.size * self.size - 1
        return self.state, r, done

def test_entropy_ac():
    """测试熵正则化AC"""
    n_states, n_actions = 16, 2
    ac = EntropyAC(n_states, n_actions)
    env = GridEnvAC()
    rewards_history = []
    
    for ep in range(100):
        s = env.reset()
        total_reward = 0
        for _ in range(50):
            exp_a = np.exp(ac.actor[s] - np.max(ac.actor[s]))
            a = np.random.choice(n_actions, p=exp_a / np.sum(exp_a))
            ns, r, done = env.step(a)
            ac.update(s, a, r, ns, done)
            s = ns
            total_reward += r
            if done: break
        rewards_history.append(total_reward)
    
    print("熵正则化AC:")
    print(f"  最终回报: {rewards_history[-1]}")
    print(f"  策略熵(状态0): {ac.entropy(0):.3f}")

test_entropy_ac()

print("\n第14章 Actor-Critic方法 - 编程题完成!")